In [43]:
import fiftyone
import fiftyone.utils.random
import pathlib
import yaml
from ultralytics import YOLO


# set up the paths, names and settings

In [44]:
# select to models parent folder
models_parent_dir = pathlib.Path.cwd().parent / "models"
datasets_parent_dir = pathlib.Path.cwd().parent / "datasets"

# choose a name for the new merged dataset
merged_dataset_name = "der_merger"

# Load settings
with open('settings.yaml') as f:
    settings = yaml.safe_load(f)

splits = ["train", "test", "val"]

# All splits must use the same classes list
classes_to_export = ["Chair", "Window", "Light", "Door"]

# if the predictions labels are not to your liking
label_remapping = {"chair": "Chair", "window": "Window", "light": "Light"}

# choose models to inference with, they must match the settings.yaml
chosen_models = ["doors", "windows", "lights"]

# setup the datasets in fiftyone

In [45]:

try:
    tmp_dataset = fiftyone.load_dataset("tmp_dataset")
except:
    tmp_dataset = fiftyone.Dataset("tmp_dataset")
tmp_dataset.clear()

# choose a dataset to add the predictions to
source_dataset = fiftyone.load_dataset("coco-2017-train-validation-test-200")
tmp_dataset.add_samples(source_dataset)

# datasets_to_add = [fiftyone.load_dataset("ebi-fused-4")]
datasets_to_add = [fiftyone.load_dataset("ebi-fused-4")]

 100% |█████████████████| 600/600 [5.1s elapsed, 0s remaining, 177.1 samples/s]      


# model inferencing on a particular subset

In [46]:

conditions = []

# Apply all models to same field, then filter once
for model_name in chosen_models:
    model = YOLO(models_parent_dir / model_name / "best.pt")
    tmp_dataset.apply_model(model, label_field=model_name)

    for class_name, threshold in settings["models"][model_name].items():
        conditions.append(
            (fiftyone.ViewField("label") == class_name) & 
            (fiftyone.ViewField("confidence") >= threshold)
        )
    if conditions:
        keep_condition = conditions[0]
        for condition in conditions[1:]:
            keep_condition |= condition
        filtered_view = tmp_dataset.filter_labels(model_name, keep_condition)
        filtered_view.merge_labels(in_field=model_name, out_field="ground_truth")


# Delete predictions field from the original dataset
# tmp_dataset.delete_sample_fields("predictions")
tmp_dataset.save()

 100% |█████████████████| 600/600 [6.9s elapsed, 0s remaining, 91.3 samples/s]       
 100% |█████████████████| 600/600 [7.3s elapsed, 0s remaining, 82.5 samples/s]       
 100% |█████████████████| 600/600 [7.4s elapsed, 0s remaining, 84.3 samples/s]       


In [47]:
# session = fiftyone.launch_app(tmp_dataset)

# merge datasets

In [48]:
datasets_to_add.append(tmp_dataset)

size = 0

for dataset in datasets_to_add:
    size += len(dataset)


try:
    merged_dataset  = fiftyone.load_dataset(f"{merged_dataset_name}-{size}")
except:
    merged_dataset  = fiftyone.Dataset(f"{merged_dataset_name}-{size}")
merged_dataset .clear()
print(f"dataset name: {merged_dataset_name}-{size}")

for dataset in datasets_to_add:
    print(f"adding {len(dataset)} samples from {dataset.name}")
    merged_dataset.merge_samples(dataset)

merged_dataset.save()
print(merged_dataset)

dataset name: der_merger-3998
adding 3398 samples from ebi-fused-4
adding 600 samples from tmp_dataset
Name:        der_merger-3998
Media type:  image
Num samples: 3998
Persistent:  False
Tags:        []
Sample fields:
    id:               fiftyone.core.fields.ObjectIdField
    filepath:         fiftyone.core.fields.StringField
    tags:             fiftyone.core.fields.ListField(fiftyone.core.fields.StringField)
    metadata:         fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.metadata.ImageMetadata)
    created_at:       fiftyone.core.fields.DateTimeField
    last_modified_at: fiftyone.core.fields.DateTimeField
    ground_truth:     fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.labels.Detections)
    doors:            fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.labels.Detections)
    windows:          fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.labels.Detections)
    lights:           fiftyone.core.fields.EmbeddedDocumentField(fiftyo

# some tag renaming

In [49]:
# Find all samples tagged with "validation" and retag them as "val"
validation_samples = merged_dataset.match_tags("validation")
validation_samples.tag_samples("val")
validation_samples.untag_samples("validation")


# redistribute splits

In [50]:

print(f"Splits before redistribution: {merged_dataset.count_sample_tags()}")
merged_dataset.untag_samples(splits)
fiftyone.utils.random.random_split(merged_dataset, {"train": 0.7, "test": 0.2, "val": 0.1})
print(f"Splits after redistribution: {merged_dataset.count_sample_tags()}")

Splits before redistribution: {'train': 2920, 'test': 543, 'val': 535}
Splits after redistribution: {'test': 799, 'val': 400, 'train': 2799}


# some class renaming

In [52]:
# session = fiftyone.Session(merged_dataset)

In [53]:
label_fields = merged_dataset._get_label_fields()
for field in label_fields:
    view = merged_dataset.map_labels(field, label_remapping)
    merged_dataset = view
    

In [56]:

merged_dataset = merged_dataset.take(400)

export_dir = datasets_parent_dir / "multiclass" / f"{merged_dataset_name}-{len(merged_dataset)}"

# Export the splits
for split in splits:
    split_view = merged_dataset.match_tags(split)
    split_view.export(
        export_dir=str(export_dir),
        dataset_type=fiftyone.types.YOLOv5Dataset,
        label_field="ground_truth",
        # overwrite=True,
        split=split,
        classes=classes_to_export,
        progress=True,
    )


 100% |█████████████████| 294/294 [465.4ms elapsed, 0s remaining, 631.8 samples/s]      
Directory '/home/freeze/GIT/onemanstreasure/datasets/multiclass/der_merger-400' already exists; export will be merged with existing files
 100% |███████████████████| 70/70 [152.8ms elapsed, 0s remaining, 458.1 samples/s]     
Directory '/home/freeze/GIT/onemanstreasure/datasets/multiclass/der_merger-400' already exists; export will be merged with existing files
 100% |███████████████████| 36/36 [84.1ms elapsed, 0s remaining, 428.0 samples/s] 
